<div class="alert alert-block alert-info">This Jupyter Notebook is derived from the course <strong>"LLMs as Judges for Search" by OpenSource Connections.</strong>
    
Check out https://opensourceconnections.com/training/ for the full course and other classes.</div>



# Lab: Starting from human evaluations


In this lab we will go through some code that will compare the LLM output with human ratings. This, or comparing with judgments from behavioural tracking should be your starting point.

We will use Cohen's Kappa for comparing LLM judgments with human judgments (https://en.wikipedia.org/wiki/Cohen%27s_kappa - note that you should use Fleiss' Kappa if you are comparing more than two raters).


## Imports

In [1]:
import pandas as pd
import os
import json
import openai
from openai import OpenAI
from sklearn.metrics import cohen_kappa_score
from getpass import getpass

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")
api_key = os.environ.get('OPENAI_API_KEY')

Enter your OpenAI API Key:  ········


In [3]:
pd.set_option('display.max_colwidth', None)

## Product Data to Judge

Look at the following dataset. It contains search result data for the queries 'duct tape', 'iphone xr cool cases for teenage girls' and 'laptop' together with some human ratings that came from the ESCI dataset. How do they compare to the ratings that we'll get from the LLM?



In [4]:
df_products = pd.read_json('course/evals/data/esci-mini.json')

df_products.head(5)

,query,product_id,esci_label,binary_label,label,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,duct tape,B00DJWTGAG,E,1,3,"3M 2979 Multi-Use Duct Tape, Silver, 1.88 in x 60 yd x 7 mil, 1 Pack, Temporary Repair, Patching, Tabbing, Capping Pipe, Marking, Labeling",None,"Commercial grade – Silver duct tape resists curling and tears off roll cleanly for light use in professional MRO/construction applications\nFlexible adhesive – Aggressive synthetic rubber adhesive sticks immediately to a wide variety of surfaces\nMulti-use applications– Great for patching, tabbing, light duty bundling, capping pipe, marking, labeling or temporary repair\nPriced for economy – This multipurpose contractor-grade tape is economically priced to best provide a quality temporary solution for light duties",3M,Silver,us
1,duct tape,B078M21NYH,E,1,3,"Craftzilla Rainbow Colored Duct Tape — 6 Bright Colors — 10 Yards x 2 Inch — No Residue, Tear by Hand & Waterproof — Great for Arts & Crafts, Color-Coding, and DIY Projects","This multi purpose rainbow set of duct tape colors and patterns is great for Students, Teachers, Parents, Artists, and Professionals. Use this colored duct tape for crafts and art projects that can be done in the classroom, studio, kitchen, home, or garage. The colorful duct tape rips nicely and cleanly so it can easily be used and applied to different surfaces. This fun duct tape bulk pack comes with 6 fun duct tape craft rolls - each roll measuring 10 yards of 2 inch duct tape. This includes 6 rolls in an assortment of bright neon duct tape colors (not fluorescent) including: Pink, Orange, Yellow, Green, Blue, Purple. This colorful tape set can be made into DIY arts and crafts projects such as the construction of forts, making duct tape wallets, keychains, origami, personalizing and decorating journals, notebooks, and luggages. This is also great for color-coding, labeling, and organizing use making it a fun addition to your DIY kits. Whenever you move or need storage organization, use these duct tape assorted colors to make labels for boxes and storage bins and put corresponding duct tape rainbow color tags to each room. Discover creative possibilities with these duct tape arts and crafts kits. Art is limitless so use this duct tape for boys, girls, adults, artists, and professionals!","Vivid Vibrance – Enjoy eye-catching brightness and a full rainbow of color options. Make labels, gifts, and statements—your multi purpose color duct tape pack from Craftzilla is both your palette and canvas.\nTearable and Easy to Clean – Assemble rainbow duct tape kits at home, in school, and on vacation. Your craft tapes are easy for tiny hands to tear for their craft and construction projects, and no hassle to peel off for easy cleanup with no residue.\nYou’re on a Roll – And you’ve got plenty left! With 60 total yards of wonderful hues and easy-tear workability, you can use your colored duct tape variety pack for project after project.\nCommunicate With Color – Apply your multi color duct tape anywhere you need to be heard without saying a word. Effortlessly label moving boxes, organize unruly TV cabling, and mark social-distancing spots.\nStick With Us – Count on Craftzilla for colored tape duct so fun, bright, and inspiring you won’t want to put it down. Your tape set is backed by our commitment to your colorful and crafty success.",Craftzilla,"Rainbow - Pink, Orange, Yellow, Green, Blue, Violet",us
2,duct tape,B0021L9MVO,I,0,0,"Duck HD Clear Heavy Duty Packing Tape, 1.88 Inch x 109 Yards, 6 Rolls (299016)",None,"Heavy duty for a strong and secure hold to keep your valuables safe while moving, shipping or in storage\nOffers wide temperature range performance for shipping and storage in hot or cold temperatures\nAdhesive bond strengthens over time for a long-lasting hold on boxes, perfect for storage\nCrystal clear to the core for a professional look on boxes or taping address labels\nMeets postal

In [5]:
SYSTEM_PROMPT = """

You are an expert relevance judgment system. Your task is to assess the relevance of a given document to a specific user query.
The document is a product title.

Provide a relevance rating by assigning the label 'relevant' or 'not relevant' as property 'relevance' of an object in JSON format.

"""

In [6]:
def make_user_prompt(query, doc_title):
    return f"""
    
User Query: {query}

Document title:\n{doc_title}

Based on the above, provide a relevance judgment."""



In [7]:
CLIENT = OpenAI(api_key=api_key)

def evaluate(query, doc_id, doc_title, response_judgment_property='relevance', client=CLIENT, system_prompt=SYSTEM_PROMPT):
    
    try:
        # Send request to OpenAI API
        # Using generate_content and specifying the response_mime_type for JSON output
       
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "developer", "content": system_prompt},
                {
                    "role": "user",
                    "content": make_user_prompt(query, doc_title),
                },
            ],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        
        # Parse the JSON response
        json_output = response.choices[0].message.content
        judgment = json.loads(json_output)
        #print(judgment)
        
        return judgment[response_judgment_property]
        
        

    except Exception as e:
        print(f"Error processing OpenAI judgment for Query: '{query[:30]}...', Doc ID: {doc_id}: {e}")
        raise e
        #return None
        

In [8]:
df_products['judgment'] = df_products \
    .apply(lambda row: evaluate(query=row['query'], doc_id=row['product_id'], doc_title=row['product_title']), axis=1, result_type="expand")
df_products[['query', 'product_title', 'judgment', 'binary_label']].head(20)


,query,product_title,judgment,binary_label
0,duct tape,"3M 2979 Multi-Use Duct Tape, Silver, 1.88 in x 60 yd x 7 mil, 1 Pack, Temporary Repair, Patching, Tabbing, Capping Pipe, Marking, Labeling",relevant,1
1,duct tape,"Craftzilla Rainbow Colored Duct Tape — 6 Bright Colors — 10 Yards x 2 Inch — No Residue, Tear by Hand & Waterproof — Great for Arts & Crafts, Color-Coding, and DIY Projects",relevant,1
2,duct tape,"Duck HD Clear Heavy Duty Packing Tape, 1.88 Inch x 109 Yards, 6 Rolls (299016)",not relevant,0
3,duct tape,"Gorilla 6035180 Duct Tape, 1 Pack, Black",relevant,1
4,duct tape,"Duck Max Strength 240201 Duct Tape, 1-Pack 1.88 Inch x 45 Yard Silver",relevant,1
5,duct tape,"Duck 299002 Brand EZ Start Packaging Tape Clear, 60 Yards/Roll (Single)",not relevant,0
6,duct tape,Electriduct D-2 Rubber Duct Cord Cover - 5 Feet Black Floor Cable Protector,not relevant,0
7,duct tape,"ChromaLabel 1 Inch Clean Remove Color-Code Tape, 500 Inch Roll, Red",not relevant,1
8,duct tape,"Scotch Heavy Duty Shipping Packaging Tape, 1.88"" x 54.6 Yards, 3"" Core, Clear, Great for Packing, Shipping & Moving, 1 Roll, Dispensered (3850-RD)",not relevant,0
9,duct tape,JVCC J90 Low Gloss Gaffer-Style Duct Tape: 4 in. (96mm Actual) x 75 ft. (Black),relevant,0


## Comparison
We turn our LLM judgments into numerical judgments for easier comparison. Then we inspect the documents where the LLM judge and the human rater disagree. 

In [9]:
df_products['binary_judgment'] = df_products['judgment'].apply(lambda j: 1 if j == 'relevant' else 0)

df_products[df_products['binary_judgment'] != df_products['binary_label']][['query', 'product_title', 'judgment', 'binary_label', 'binary_judgment']]

,query,product_title,judgment,binary_label,binary_judgment
7,duct tape,"ChromaLabel 1 Inch Clean Remove Color-Code Tape, 500 Inch Roll, Red",not relevant,1,0
9,duct tape,JVCC J90 Low Gloss Gaffer-Style Duct Tape: 4 in. (96mm Actual) x 75 ft. (Black),relevant,0,1
13,iphone xr cool cases for teenage girls,"iPhone Xs Case for Girls, YeLoveHaw Flexible Soft Slim Fit Full-Around Protective Cute Shell Phone Case Cover with Purple Floral and Gray Leaves Pattern for iPhone X/XS 5.8 Inch (Pink Flowers)",not relevant,1,0
20,laptop,Broonel Black Invisible Lightweight Laptop Computer Stand - Compatible with The Lenovo 15.6 Inch Intel Ci3 8GB 1TB Laptop,relevant,0,1
21,laptop,Broonel Black Invisible Lightweight Laptop Computer Stand - Compatible with The Lenovo (15.6 inch HD) Notebook (AMD A4-9125 2x2.6 GHz,relevant,0,1


## Quantifying the agreement
There are muliple ways to quantify the agreement or disagreement between the two raters. A simple approach is to just look at the percentage of agreement amongst all ratings:

In [10]:
count_agreement = df_products[df_products['binary_judgment'] == df_products['binary_label']].shape[0]
agreement_pct = (count_agreement / df_products.shape[0]) * 100
print(f"Agreement between human and LLM: {agreement_pct:.2f} %")

Agreement between human and LLM: 83.33 %


In [11]:
# Per query
df_products.groupby(by='query').apply(
    lambda g: g[g['binary_label'] == g['binary_judgment']].shape[0] / g.shape[0]
).to_frame(name='agree %').reset_index()

/tmp/ipykernel_249/2988033462.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_products.groupby(by='query').apply(


,query,agree %
0,duct tape,0.8
1,iphone xr cool cases for teenage girls,0.9
2,laptop,0.8


Measuring agreement using Cohen's Kappa:

In [12]:
kappa = cohen_kappa_score(df_products['binary_label'], df_products['binary_judgment'])
print(f"Cohen's Kappa: {kappa:.2f}")

Cohen's Kappa: 0.65


In [13]:
# Per query
df_kappa = df_products.groupby(by='query').apply(
    lambda g: cohen_kappa_score(g['binary_label'], g['binary_judgment'])
).to_frame(name='kappa').reset_index()

df_kappa

/tmp/ipykernel_249/603299009.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_kappa = df_products.groupby(by='query').apply(


,query,kappa
0,duct tape,0.600000
1,iphone xr cool cases for teenage girls,0.800000
2,laptop,0.411765


## Optimizing the prompt towards a better agreement between the LLM and human raters

In [14]:
SYSTEM_PROMPT_BETTER = """

You are an expert relevance judgment system. Your task is to assess the relevance of a given document to a specific user query.
The document is a product title.

Provide a relevance rating by assigning the label 'relevant' or 'not relevant' as property 'relevance' of an object in JSON format.
In addition to the rating, provide a concise reasoning for your judgment.

If the product is an accessory while the query's intent is for the main product, use label 'not relevant'.

First reason, then judge based on the reasoning.

Your output MUST be in the following JSON format:
{
  "reasoning": "A concise explanation for the rating.",
  "relevance": <relevance_rating>
}




"""

In [15]:
# Re-read data frame
df_products = pd.read_json('course/evals/data/esci-mini.json')

df_products.head(5)

,query,product_id,esci_label,binary_label,label,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,duct tape,B00DJWTGAG,E,1,3,"3M 2979 Multi-Use Duct Tape, Silver, 1.88 in x 60 yd x 7 mil, 1 Pack, Temporary Repair, Patching, Tabbing, Capping Pipe, Marking, Labeling",None,"Commercial grade – Silver duct tape resists curling and tears off roll cleanly for light use in professional MRO/construction applications\nFlexible adhesive – Aggressive synthetic rubber adhesive sticks immediately to a wide variety of surfaces\nMulti-use applications– Great for patching, tabbing, light duty bundling, capping pipe, marking, labeling or temporary repair\nPriced for economy – This multipurpose contractor-grade tape is economically priced to best provide a quality temporary solution for light duties",3M,Silver,us
1,duct tape,B078M21NYH,E,1,3,"Craftzilla Rainbow Colored Duct Tape — 6 Bright Colors — 10 Yards x 2 Inch — No Residue, Tear by Hand & Waterproof — Great for Arts & Crafts, Color-Coding, and DIY Projects","This multi purpose rainbow set of duct tape colors and patterns is great for Students, Teachers, Parents, Artists, and Professionals. Use this colored duct tape for crafts and art projects that can be done in the classroom, studio, kitchen, home, or garage. The colorful duct tape rips nicely and cleanly so it can easily be used and applied to different surfaces. This fun duct tape bulk pack comes with 6 fun duct tape craft rolls - each roll measuring 10 yards of 2 inch duct tape. This includes 6 rolls in an assortment of bright neon duct tape colors (not fluorescent) including: Pink, Orange, Yellow, Green, Blue, Purple. This colorful tape set can be made into DIY arts and crafts projects such as the construction of forts, making duct tape wallets, keychains, origami, personalizing and decorating journals, notebooks, and luggages. This is also great for color-coding, labeling, and organizing use making it a fun addition to your DIY kits. Whenever you move or need storage organization, use these duct tape assorted colors to make labels for boxes and storage bins and put corresponding duct tape rainbow color tags to each room. Discover creative possibilities with these duct tape arts and crafts kits. Art is limitless so use this duct tape for boys, girls, adults, artists, and professionals!","Vivid Vibrance – Enjoy eye-catching brightness and a full rainbow of color options. Make labels, gifts, and statements—your multi purpose color duct tape pack from Craftzilla is both your palette and canvas.\nTearable and Easy to Clean – Assemble rainbow duct tape kits at home, in school, and on vacation. Your craft tapes are easy for tiny hands to tear for their craft and construction projects, and no hassle to peel off for easy cleanup with no residue.\nYou’re on a Roll – And you’ve got plenty left! With 60 total yards of wonderful hues and easy-tear workability, you can use your colored duct tape variety pack for project after project.\nCommunicate With Color – Apply your multi color duct tape anywhere you need to be heard without saying a word. Effortlessly label moving boxes, organize unruly TV cabling, and mark social-distancing spots.\nStick With Us – Count on Craftzilla for colored tape duct so fun, bright, and inspiring you won’t want to put it down. Your tape set is backed by our commitment to your colorful and crafty success.",Craftzilla,"Rainbow - Pink, Orange, Yellow, Green, Blue, Violet",us
2,duct tape,B0021L9MVO,I,0,0,"Duck HD Clear Heavy Duty Packing Tape, 1.88 Inch x 109 Yards, 6 Rolls (299016)",None,"Heavy duty for a strong and secure hold to keep your valuables safe while moving, shipping or in storage\nOffers wide temperature range performance for shipping and storage in hot or cold temperatures\nAdhesive bond strengthens over time for a long-lasting hold on boxes, perfect for storage\nCrystal clear to the core for a professional look on boxes or taping address labels\nMeets postal

In [16]:
df_products['judgment'] = df_products \
    .apply(lambda row: evaluate(query=row['query'], doc_id=row['product_id'], doc_title=row['product_title'], system_prompt=SYSTEM_PROMPT_BETTER), axis=1, result_type="expand")
df_products[['query', 'product_title', 'judgment', 'binary_label']].head(20)

,query,product_title,judgment,binary_label
0,duct tape,"3M 2979 Multi-Use Duct Tape, Silver, 1.88 in x 60 yd x 7 mil, 1 Pack, Temporary Repair, Patching, Tabbing, Capping Pipe, Marking, Labeling",relevant,1
1,duct tape,"Craftzilla Rainbow Colored Duct Tape — 6 Bright Colors — 10 Yards x 2 Inch — No Residue, Tear by Hand & Waterproof — Great for Arts & Crafts, Color-Coding, and DIY Projects",relevant,1
2,duct tape,"Duck HD Clear Heavy Duty Packing Tape, 1.88 Inch x 109 Yards, 6 Rolls (299016)",not relevant,0
3,duct tape,"Gorilla 6035180 Duct Tape, 1 Pack, Black",relevant,1
4,duct tape,"Duck Max Strength 240201 Duct Tape, 1-Pack 1.88 Inch x 45 Yard Silver",relevant,1
5,duct tape,"Duck 299002 Brand EZ Start Packaging Tape Clear, 60 Yards/Roll (Single)",not relevant,0
6,duct tape,Electriduct D-2 Rubber Duct Cord Cover - 5 Feet Black Floor Cable Protector,not relevant,0
7,duct tape,"ChromaLabel 1 Inch Clean Remove Color-Code Tape, 500 Inch Roll, Red",not relevant,1
8,duct tape,"Scotch Heavy Duty Shipping Packaging Tape, 1.88"" x 54.6 Yards, 3"" Core, Clear, Great for Packing, Shipping & Moving, 1 Roll, Dispensered (3850-RD)",not relevant,0
9,duct tape,JVCC J90 Low Gloss Gaffer-Style Duct Tape: 4 in. (96mm Actual) x 75 ft. (Black),relevant,0


In [17]:
df_products['binary_judgment'] = df_products['judgment'].apply(lambda j: 1 if j == 'relevant' else 0)

df_products[df_products['binary_judgment'] != df_products['binary_label']][['query', 'product_title', 'judgment', 'binary_label', 'binary_judgment']]

,query,product_title,judgment,binary_label,binary_judgment
7,duct tape,"ChromaLabel 1 Inch Clean Remove Color-Code Tape, 500 Inch Roll, Red",not relevant,1,0
9,duct tape,JVCC J90 Low Gloss Gaffer-Style Duct Tape: 4 in. (96mm Actual) x 75 ft. (Black),relevant,0,1
13,iphone xr cool cases for teenage girls,"iPhone Xs Case for Girls, YeLoveHaw Flexible Soft Slim Fit Full-Around Protective Cute Shell Phone Case Cover with Purple Floral and Gray Leaves Pattern for iPhone X/XS 5.8 Inch (Pink Flowers)",not relevant,1,0
26,laptop,"Broonel Grey Invisible Lightweight Laptop Computer Stand - Compatible with The Acer Aspire 1, 14"" Full HD, Intel Celeron N3450",not relevant,1,0


In [18]:
count_agreement = df_products[df_products['binary_judgment'] == df_products['binary_label']].shape[0]
agreement_pct = (count_agreement / df_products.shape[0]) * 100
print(f"Agreement between human and LLM: {agreement_pct:.2f} %")

Agreement between human and LLM: 86.67 %


In [19]:
# Per query
df_products.groupby(by='query').apply(
    lambda g: g[g['binary_label'] == g['binary_judgment']].shape[0] / g.shape[0]
).to_frame(name='agree %').reset_index()

/tmp/ipykernel_249/2988033462.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_products.groupby(by='query').apply(


,query,agree %
0,duct tape,0.8
1,iphone xr cool cases for teenage girls,0.9
2,laptop,0.9


In [20]:
kappa = cohen_kappa_score(df_products['binary_label'], df_products['binary_judgment'])
print(f"Cohen's Kappa: {kappa:.2f}")

Cohen's Kappa: 0.73
